# Binary Classification — Default Prediction (`gb`)

**Goal:** Build a binary classification model to predict the target variable `gb` (0 = no default, 1 = default).

**Data:** `train_df.csv` — ~26,824 rows × 554 columns.
- `id` — entity identifier (several rows per id — panel data)
- `gb` — binary target (0 or 1)
- `cat_*` — categorical features (integer codes)
- `num_*` — numerical features (many missing values)

**Models:**
1. Logistic Regression (linear baseline)
2. CatBoost (gradient boosting with native categorical support)
3. LightGBM (histogram-based gradient boosting)

---

## Notebook Structure

1. Imports & Configuration
2. Load Data & Initial Overview
3. Missing Values Analysis
4. Numerical Feature Distributions & Statistical Tests
5. Multivariate Analysis (Correlations, Cardinality, Entity Activity)
6. Data Cleaning & Feature Selection
7. Common Preprocessing (applied once to the dataset)
8. Train/Val Split (StratifiedGroupKFold)
9. Preprocessing Functions for Logistic Regression vs Trees
10. Logistic Regression (5-fold CV)
11. CatBoost (5-fold CV)
12. LightGBM (5-fold CV)
13. Model Comparison & Final Plots

---
## 1. Imports & Configuration

In [ ]:
import warnings
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import chi2_contingency, mannwhitneyu

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve

from catboost import CatBoostClassifier, Pool
import lightgbm as lgb

warnings.filterwarnings("ignore")

# ── Constants ─────────────────────────────────────────────────────────────────
DATA_PATH    = "train_df.csv"
TARGET_COL   = "gb"
ID_COL       = "id"
RANDOM_STATE = 42
N_SPLITS     = 5

# Color palette for plots
PALETTE = {
    "primary":   "#2E86AB",
    "secondary": "#A23B72",
    "accent":    "#F18F01",
    "pos":       "#C73E1D",
    "neg":       "#3A7D44",
}

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette([PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]])
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 13, "axes.labelsize": 11})
%matplotlib inline

np.random.seed(RANDOM_STATE)
print("All imports loaded successfully.")

**Why:** We load all libraries upfront so the notebook is self-contained — no external `.py` files needed. `warnings.filterwarnings("ignore")` suppresses convergence warnings from sklearn that would clutter the output. We set a fixed `RANDOM_STATE` for reproducibility across all models and splits.

---
## 2. Load Data & Initial Overview

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique entities (id): {df[ID_COL].nunique():,}")
print(f"Average rows per entity: {df.shape[0] / df[ID_COL].nunique():.1f}")
print(f"\nTarget distribution ({TARGET_COL}):")
print(df[TARGET_COL].value_counts())
print(f"\nPositive class rate: {df[TARGET_COL].mean():.2%}")

# Detect feature types
cat_cols = sorted([c for c in df.columns if c.startswith("cat_")])
num_cols = sorted([c for c in df.columns if c.startswith("num_")])
print(f"\nFeatures: {len(cat_cols)} categorical, {len(num_cols)} numerical")

df.head()

**Why:** Before any modeling we need to understand the data dimensions and target balance. The ~2.2% positive rate tells us this is a heavily imbalanced dataset, which means:
- We cannot use accuracy as a metric (a naive "always predict 0" model gets ~97.8% accuracy).
- We must use PR-AUC (Average Precision) as the primary metric and apply class weighting.
- The panel structure (~5 rows per entity) means we must split by `id` groups to avoid data leakage.

---
## 3. Missing Values Analysis

In [ ]:
# Overall missing percentage per column
miss_pct = df[num_cols + cat_cols].isnull().mean().sort_values(ascending=False)
miss_pct_nonzero = miss_pct[miss_pct > 0]

print(f"Columns with missing values: {len(miss_pct_nonzero)} out of {len(num_cols + cat_cols)}")
print(f"Columns with 100% missing:   {(miss_pct == 1.0).sum()}")
print(f"Columns with >80% missing:   {(miss_pct > 0.8).sum()}")
print(f"Columns with >50% missing:   {(miss_pct > 0.5).sum()}")

# Top 20 most missing columns
fig, ax = plt.subplots(figsize=(10, 6))
miss_pct_nonzero.head(30).plot(kind="barh", ax=ax, color=PALETTE["primary"])
ax.set_xlabel("Fraction missing")
ax.set_title("Top 30 Columns by Missing Value Rate")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# Columns that are 100% missing — these will be dropped
all_nan_cols = miss_pct[miss_pct == 1.0].index.tolist()
print(f"\nColumns to drop (100% NaN): {all_nan_cols}")

**Why:** Understanding the missing data pattern is critical before imputation. Columns that are 100% NaN carry zero information and must be dropped. We also want to identify MNAR (Missing Not At Random) columns — where the *fact* of missing itself is predictive of the target. These will get binary `is_missing` indicator features later (especially useful for logistic regression).

In [ ]:
# MNAR detection: is the missing rate different between target=0 and target=1?
mnar_results = []
for col in num_cols:
    if df[col].isnull().mean() == 0 or df[col].isnull().mean() == 1.0:
        continue
    miss_rate_0 = df.loc[df[TARGET_COL] == 0, col].isnull().mean()
    miss_rate_1 = df.loc[df[TARGET_COL] == 1, col].isnull().mean()
    diff = abs(miss_rate_1 - miss_rate_0)
    if diff > 0.05:  # at least 5 percentage point difference
        mnar_results.append({"column": col, "miss_rate_gb0": miss_rate_0, "miss_rate_gb1": miss_rate_1, "diff": diff})

mnar_df = pd.DataFrame(mnar_results).sort_values("diff", ascending=False)
MNAR_COLS = mnar_df["column"].tolist()
print(f"MNAR columns detected (missing rate differs by >5pp between classes): {len(MNAR_COLS)}")
display(mnar_df.head(15))

**Why:** If a column's missing rate is significantly different between defaulters (gb=1) and non-defaulters (gb=0), then the missingness itself is informative — this is called MNAR (Missing Not At Random). For logistic regression we will add binary `is_missing` indicator features for these columns. Tree-based models can learn missing patterns natively, so they don't need these indicators.

---
## 4. Numerical Feature Distributions & Statistical Tests

In [ ]:
# Mann-Whitney U test: which numerical features differ significantly between classes?
mw_results = []
valid_num_cols = [c for c in num_cols if df[c].notna().any() and df[c].notna().sum() > 100]

for col in valid_num_cols:
    vals_0 = df.loc[df[TARGET_COL] == 0, col].dropna()
    vals_1 = df.loc[df[TARGET_COL] == 1, col].dropna()
    if len(vals_0) < 10 or len(vals_1) < 10:
        continue
    stat, pval = mannwhitneyu(vals_0, vals_1, alternative="two-sided")
    mw_results.append({"column": col, "p_value": pval, "median_gb0": vals_0.median(), "median_gb1": vals_1.median()})

mw_df = pd.DataFrame(mw_results).sort_values("p_value")
significant = mw_df[mw_df["p_value"] < 0.05]
print(f"Numerical features significantly different between classes (p<0.05): {len(significant)} / {len(mw_df)}")
display(significant.head(15))

**Why:** The Mann-Whitney U test is a non-parametric test that checks if the distribution of each numerical feature differs between the two target classes. Features with very low p-values are statistically associated with default — they are the most promising predictors. We use Mann-Whitney (not t-test) because financial data is typically heavily skewed and non-normal.

In [ ]:
# Skewness and outlier analysis
skew_series = df[valid_num_cols].skew().sort_values(ascending=False)
print(f"Median skewness: {skew_series.median():.2f}")
print(f"Features with |skewness| > 5: {(skew_series.abs() > 5).sum()}")
print(f"Features with |skewness| > 10: {(skew_series.abs() > 10).sum()}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(skew_series.clip(-20, 20), bins=50, color=PALETTE["primary"], edgecolor="white")
ax.set_xlabel("Skewness")
ax.set_ylabel("Number of features")
ax.set_title("Distribution of Feature Skewness (clipped to [-20, 20])")
ax.axvline(0, color="red", ls="--", lw=1)
plt.tight_layout()
plt.show()

**Why:** Highly skewed features cause problems for logistic regression (the coefficients become unstable). This confirms that we need `RobustScaler` (not `StandardScaler`) for logistic regression — it uses median and IQR instead of mean and std, so it is resistant to outliers. Tree-based models (CatBoost, LightGBM) are naturally immune to skewness and outliers, so they don't need scaling.

In [ ]:
# Chi-squared test for categorical features vs target
chi2_results = []
for col in cat_cols:
    ct = pd.crosstab(df[col], df[TARGET_COL])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        continue
    chi2, pval, dof, _ = chi2_contingency(ct)
    chi2_results.append({"column": col, "chi2": chi2, "p_value": pval, "dof": dof, "cardinality": df[col].nunique()})

chi2_df = pd.DataFrame(chi2_results).sort_values("p_value")
chi2_sig = chi2_df[chi2_df["p_value"] < 0.05]
print(f"Categorical features significantly associated with target (p<0.05): {len(chi2_sig)} / {len(chi2_df)}")
display(chi2_df.head(15))

**Why:** The Chi-squared test of independence checks whether each categorical feature is associated with the target variable. Features with low p-values have a statistically significant relationship with default. This complements the Mann-Whitney test (which covers numerical features). Together, they give us a full picture of which features carry signal.

---
## 5. Multivariate Analysis

In [ ]:
# Correlation matrix of top-20 most significant numerical features
top_num = mw_df.head(20)["column"].tolist()
corr = df[top_num].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".1f", ax=ax, square=True, linewidths=0.5)
ax.set_title("Correlation Matrix — Top 20 Significant Numerical Features")
plt.tight_layout()
plt.show()

**Why:** Highly correlated features (|r| > 0.95) are redundant and harmful for logistic regression — they cause multicollinearity, inflating coefficient variance and making the model unstable. We will remove one feature from each highly-correlated pair during data cleaning. Tree models are less affected but still benefit from reduced feature redundancy (faster training, less overfitting).

In [ ]:
# Categorical feature cardinality
cardinality = df[cat_cols].nunique().sort_values(ascending=False)
print(f"Max cardinality: {cardinality.max()} (column: {cardinality.idxmax()})")
print(f"Features with cardinality > 20: {(cardinality > 20).sum()}")
print(f"Features with cardinality <= 5: {(cardinality <= 5).sum()}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cardinality.clip(0, 100), bins=50, color=PALETTE["secondary"], edgecolor="white")
ax.set_xlabel("Number of unique values")
ax.set_ylabel("Number of features")
ax.set_title("Categorical Feature Cardinality (clipped at 100)")
plt.tight_layout()
plt.show()

**Why:** Cardinality affects the encoding strategy. CatBoost handles high-cardinality categoricals natively (ordered target encoding). LightGBM also handles them well. But logistic regression needs label encoding + scaling for all categoricals — high cardinality doesn't cause one-hot explosion because we use label encoding, not one-hot.

In [ ]:
# Entity activity: rows per ID vs default rate
entity_stats = df.groupby(ID_COL).agg(
    n_rows=(TARGET_COL, "count"),
    default_rate=(TARGET_COL, "mean"),
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(entity_stats["n_rows"], bins=30, color=PALETTE["primary"], edgecolor="white")
axes[0].set_xlabel("Rows per entity")
axes[0].set_ylabel("Number of entities")
axes[0].set_title("Entity Activity Distribution")

axes[1].scatter(entity_stats["n_rows"], entity_stats["default_rate"],
                alpha=0.3, s=10, color=PALETTE["secondary"])
axes[1].set_xlabel("Rows per entity")
axes[1].set_ylabel("Default rate")
axes[1].set_title("Entity Activity vs Default Rate")

plt.tight_layout()
plt.show()

print(f"Entity row count stats: mean={entity_stats['n_rows'].mean():.1f}, "
      f"median={entity_stats['n_rows'].median():.0f}, "
      f"max={entity_stats['n_rows'].max()}")

**Why:** With panel data (multiple rows per entity), we need to understand the structure before splitting. If an entity appears in both train and validation, the model memorizes entity-specific patterns — this is **data leakage**. That's why we use `StratifiedGroupKFold` (splitting by `id` groups) instead of simple `train_test_split`.

---
## 6. Data Cleaning & Feature Selection

In [ ]:
# ── Step 1: Drop 100% NaN columns ────────────────────────────────────────────
cols_before = len(df.columns)
df = df.drop(columns=all_nan_cols, errors="ignore")
print(f"Dropped {cols_before - len(df.columns)} columns with 100% missing values.")

# Update column lists
cat_cols = sorted([c for c in df.columns if c.startswith("cat_")])
num_cols = sorted([c for c in df.columns if c.startswith("num_")])

# ── Step 2: Drop exact duplicate columns ─────────────────────────────────────
feature_cols = cat_cols + num_cols
dup_groups = df[feature_cols].T.duplicated(keep="first")
dup_cols = dup_groups[dup_groups].index.tolist()
df = df.drop(columns=dup_cols, errors="ignore")
print(f"Dropped {len(dup_cols)} exact duplicate columns.")

cat_cols = sorted([c for c in df.columns if c.startswith("cat_")])
num_cols = sorted([c for c in df.columns if c.startswith("num_")])

# ── Step 3: Drop quasi-constant features (variance < 0.01) ───────────────────
low_var = []
for col in num_cols:
    if df[col].dropna().var() < 0.01:
        low_var.append(col)
for col in cat_cols:
    # For categoricals: if one value represents >99% of rows
    if df[col].value_counts(normalize=True).iloc[0] > 0.99:
        low_var.append(col)

df = df.drop(columns=low_var, errors="ignore")
print(f"Dropped {len(low_var)} quasi-constant features.")

cat_cols = sorted([c for c in df.columns if c.startswith("cat_")])
num_cols = sorted([c for c in df.columns if c.startswith("num_")])

# ── Step 4: Remove highly correlated numerical features (|r| > 0.95) ─────────
corr_matrix = df[num_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
df = df.drop(columns=high_corr_cols, errors="ignore")
print(f"Dropped {len(high_corr_cols)} highly correlated numerical features (|r|>0.95).")

# Final feature lists
cat_cols = sorted([c for c in df.columns if c.startswith("cat_")])
num_cols = sorted([c for c in df.columns if c.startswith("num_")])
FINAL_FEATURES = cat_cols + num_cols

print(f"\n=== Final feature set: {len(cat_cols)} categorical + {len(num_cols)} numerical = {len(FINAL_FEATURES)} total ===")

**Why:** We apply four cleaning steps, each removing a different type of useless feature:
1. **100% NaN** — zero information, would crash imputers.
2. **Duplicate columns** — identical columns waste computation without adding signal.
3. **Quasi-constant** — features with almost no variance can't discriminate between classes.
4. **Highly correlated pairs** — especially critical for logistic regression where multicollinearity inflates standard errors and makes coefficients unreliable.

This is done **before** the train/val split because these are properties of the features themselves (not data-dependent transformations that could leak). We keep this as a simple, flat sequence — no classes or pipelines.

---
## 7. Common Preprocessing (applied once to the whole dataset)

In [ ]:
# ── Common preprocessing applied to the whole dataset ────────────────────────
# This only includes operations that don't leak information from val into train:
#   - Casting categorical columns to int
#   - Extracting X, y, and groups

# Make sure categoricals are int-typed (they are integer codes already)
for col in cat_cols:
    df[col] = df[col].fillna(-1).astype(int)

X_all = df[FINAL_FEATURES].copy()
y_all = df[TARGET_COL].values
groups_all = df[ID_COL].values

# Identify which final features are MNAR (for LR missing indicators)
final_num = [c for c in FINAL_FEATURES if c.startswith("num_")]
final_cat = [c for c in FINAL_FEATURES if c.startswith("cat_")]
final_mnar = [c for c in MNAR_COLS if c in final_num]

print(f"X_all shape: {X_all.shape}")
print(f"Final numerical:   {len(final_num)}")
print(f"Final categorical: {len(final_cat)}")
print(f"MNAR features:     {len(final_mnar)}")
print(f"Target balance: {pd.Series(y_all).value_counts(normalize=True).round(4).to_dict()}")

**Why:** We separate "safe" preprocessing (that doesn't cause leakage) from "fold-specific" preprocessing (that must be fitted on training data only). Casting categoricals to int and extracting X/y is safe to do on the full dataset. Everything else — imputation, scaling, encoding — will be done inside each fold to prevent information from the validation set leaking into the training pipeline.

---
## 8. Train/Val Split (StratifiedGroupKFold)

In [ ]:
# StratifiedGroupKFold: stratifies by target, groups by entity ID
# This ensures:
#  1. No entity appears in both train and val (prevents leakage)
#  2. Each fold has roughly the same positive class rate

entity_target = df.groupby(ID_COL)[TARGET_COL].first().reset_index()
entity_y = df[ID_COL].map(entity_target.set_index(ID_COL)[TARGET_COL]).values

sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
splits = list(sgkf.split(X_all, entity_y, groups=groups_all))

print(f"CV splits (StratifiedGroupKFold, {N_SPLITS} folds):")
for i, (tr, va) in enumerate(splits):
    n_groups_tr = len(set(groups_all[tr]))
    n_groups_va = len(set(groups_all[va]))
    overlap = set(groups_all[tr]) & set(groups_all[va])
    print(f"  Fold {i}: train={len(tr):,} rows ({n_groups_tr} entities), "
          f"val={len(va):,} rows ({n_groups_va} entities), "
          f"val positive rate={y_all[va].mean():.2%}, "
          f"id overlap={len(overlap)}")

**Why:** `StratifiedGroupKFold` is the correct splitting strategy for this dataset because:
1. **Group splitting by `id`**: Since each entity has ~5 rows, a naive random split would place some rows of the same entity in train and others in val. The model would then memorize entity-specific patterns — this is leakage, and the validation metrics would be artificially inflated.
2. **Stratification**: With only ~2.2% positive class, random splits could produce folds with very few (or zero) positives, making metrics unstable.
3. **id overlap = 0** confirms no leakage between folds.

---
## 9. Preprocessing Functions: Logistic Regression vs Trees

**Key difference:** Logistic regression needs scaled, imputed features with MNAR indicators. Tree models handle NaN and scale natively — they only need label-encoded categoricals.

In [ ]:
# ── Helper: sanitize column names (LightGBM/CatBoost don't like special chars) ──
def sanitize_names(cols):
    """Replace special characters in column names."""
    return [re.sub(r"[^\w]", "_", c) for c in cols]


# ── Helper: label encode categoricals (leakage-safe: fit on train only) ──
def encode_categoricals(X_train, X_val, cat_cols):
    """Label encoding fitted on train, applied to val."""
    X_tr, X_va = X_train.copy(), X_val.copy()
    for col in cat_cols:
        if col not in X_tr.columns:
            continue
        uniques = X_tr[col].astype(str).fillna("__NA__").unique()
        mapping = {v: i for i, v in enumerate(sorted(uniques))}
        X_tr[col] = X_tr[col].astype(str).fillna("__NA__").map(mapping).fillna(-1).astype(int)
        X_va[col] = X_va[col].astype(str).fillna("__NA__").map(mapping).fillna(-1).astype(int)
    return X_tr, X_va


# ── Preprocessing for LOGISTIC REGRESSION ────────────────────────────────────
def prepare_for_logreg(X_train, X_val, mnar_cols, num_cols, cat_cols):
    """
    Full preprocessing pipeline for Logistic Regression:
      1. Add binary is_missing indicators for MNAR columns
      2. Label encode categoricals (fit on train)
      3. Median imputation (fit on train)
      4. RobustScaler (fit on train)
    """
    X_tr = X_train.copy()
    X_va = X_val.copy()

    # 1. MNAR indicators
    for col in mnar_cols:
        if col in X_tr.columns:
            X_tr[f"{col}__miss"] = X_tr[col].isnull().astype(np.int8)
            X_va[f"{col}__miss"] = X_va[col].isnull().astype(np.int8)

    # 2. Label encoding
    X_tr, X_va = encode_categoricals(X_tr, X_va, cat_cols)

    # 3. Median imputation (fitted on train)
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    X_va = X_va.fillna(medians)

    # 4. RobustScaler (fitted on train)
    all_cols = X_tr.columns.tolist()
    scaler = RobustScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr), columns=all_cols, index=X_tr.index)
    X_va_s = pd.DataFrame(scaler.transform(X_va), columns=all_cols, index=X_va.index)

    return X_tr_s, X_va_s


# ── Preprocessing for TREE MODELS (CatBoost, LightGBM) ──────────────────────
def prepare_for_trees(X_train, X_val, cat_cols):
    """
    Minimal preprocessing for tree-based models:
      - Label encode categoricals (fit on train)
      - NaN values are kept as-is (CatBoost/LightGBM handle them natively)
      - No scaling needed (trees are scale-invariant)
    """
    return encode_categoricals(X_train, X_val, cat_cols)


# ── Helper: find optimal F1 threshold ────────────────────────────────────────
def find_optimal_f1(y_true, y_prob, n=200):
    """Search for the probability threshold that maximizes F1-score."""
    thresholds = np.linspace(0.001, 0.999, n)
    best_f1, best_t = 0.0, 0.5
    for t in thresholds:
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1


print("Preprocessing functions defined.")

**Why we have two separate preprocessing functions:**

**`prepare_for_logreg`** does 4 things:
1. **MNAR indicators** — logistic regression can't learn from NaN; adding `is_missing` flags captures the information that *something* is missing.
2. **Label encoding** — converts categorical strings to integers.
3. **Median imputation** — fills remaining NaN. We use median (not mean) because the data is heavily skewed.
4. **RobustScaler** — normalizes features so the L2 penalty treats all features equally. We use Robust (not Standard) because of extreme outliers.

**`prepare_for_trees`** only does label encoding:
- CatBoost and LightGBM handle NaN natively (they learn optimal split directions for missing values).
- Trees are scale-invariant — they split on thresholds, not distances.
- No need for MNAR indicators — trees can learn the missing pattern implicitly.

Both functions are **leakage-safe**: everything is fitted on training data only and applied to validation.

---
## 10. Logistic Regression (5-Fold CV)

In [ ]:
oof_lr = np.full(len(y_all), np.nan)
lr_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw = X_all.iloc[train_idx]
    X_va_raw = X_all.iloc[val_idx]
    y_tr = y_all[train_idx]
    y_va = y_all[val_idx]

    # Logistic Regression preprocessing
    X_tr_lr, X_va_lr = prepare_for_logreg(
        X_tr_raw, y_tr, X_va_raw,
        mnar_cols=final_mnar,
        num_cols=final_num,
        cat_cols=final_cat
    )

    lr = LogisticRegression(
        C=0.1,              # Strong regularization (prevents overfitting on 500+ features)
        penalty="l2",
        solver="saga",      # Supports L2 and works well with large datasets
        max_iter=300,
        class_weight="balanced",  # Automatically up-weights the rare positive class
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    lr.fit(X_tr_lr, y_tr)
    val_prob = lr.predict_proba(X_va_lr)[:, 1]
    oof_lr[val_idx] = val_prob

    pr_auc  = average_precision_score(y_va, val_prob)
    roc_auc = roc_auc_score(y_va, val_prob)
    lr_fold_metrics.append({"fold": fold_idx, "pr_auc": pr_auc, "roc_auc": roc_auc})
    print(f"  Fold {fold_idx}: PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f}")

lr_metrics_df = pd.DataFrame(lr_fold_metrics)
print(f"\nLogistic Regression — mean CV:")
print(f"  PR-AUC:  {lr_metrics_df['pr_auc'].mean():.4f} ± {lr_metrics_df['pr_auc'].std():.4f}")
print(f"  ROC-AUC: {lr_metrics_df['roc_auc'].mean():.4f} ± {lr_metrics_df['roc_auc'].std():.4f}")

**Why Logistic Regression:**
- **Interpretable baseline** — coefficients directly show each feature's contribution.
- **Fast training** — finishes in seconds even on 500+ features.
- **C=0.1** (strong regularization) because we have many features relative to positive samples (~593 positives vs 500+ features). Without regularization, the model would overfit badly.
- **class_weight="balanced"** automatically sets sample weights inversely proportional to class frequency, so the model pays ~45× more attention to the rare positive class.
- **solver="saga"** handles L2 regularization efficiently on larger datasets and supports parallelization.

---
## 11. CatBoost (5-Fold CV)

In [ ]:
oof_cb = np.full(len(y_all), np.nan)
cb_fold_metrics = []

cat_feature_indices = [FINAL_FEATURES.index(c) for c in final_cat if c in FINAL_FEATURES]

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw = X_all.iloc[train_idx]
    X_va_raw = X_all.iloc[val_idx]
    y_tr = y_all[train_idx]
    y_va = y_all[val_idx]

    # Tree preprocessing (minimal)
    X_tr_cb, X_va_cb = prepare_for_trees(X_tr_raw, X_va_raw, final_cat)

    safe_names = sanitize_names(X_tr_cb.columns.tolist())
    X_tr_cb.columns = safe_names
    X_va_cb.columns = safe_names

    train_pool = Pool(X_tr_cb, y_tr, cat_features=cat_feature_indices)
    val_pool   = Pool(X_va_cb, y_va, cat_features=cat_feature_indices)

    cb = CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False,
        auto_class_weights="Balanced",
    )
    cb.fit(train_pool, eval_set=val_pool, use_best_model=True, early_stopping_rounds=50)

    val_prob = cb.predict_proba(val_pool)[:, 1]
    oof_cb[val_idx] = val_prob

    pr_auc  = average_precision_score(y_va, val_prob)
    roc_auc = roc_auc_score(y_va, val_prob)
    cb_fold_metrics.append({"fold": fold_idx, "pr_auc": pr_auc, "roc_auc": roc_auc})
    print(f"  Fold {fold_idx}: PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f}")

cb_metrics_df = pd.DataFrame(cb_fold_metrics)
print(f"\nCatBoost — mean CV:")
print(f"  PR-AUC:  {cb_metrics_df['pr_auc'].mean():.4f} ± {cb_metrics_df['pr_auc'].std():.4f}")
print(f"  ROC-AUC: {cb_metrics_df['roc_auc'].mean():.4f} ± {cb_metrics_df['roc_auc'].std():.4f}")

CB_FEATURE_NAMES = safe_names

**Why CatBoost:**
- **Native categorical support** — uses ordered target encoding internally, which is better than label encoding or one-hot for high-cardinality categoricals.
- **Handles NaN natively** — no imputation needed, the model learns optimal split directions for missing values.
- **Robust to outliers and skewness** — tree splits are threshold-based, not distance-based.
- **`auto_class_weights="Balanced"`** — handles class imbalance automatically.
- **`early_stopping_rounds=50`** — stops training when validation AUC stops improving, preventing overfitting.
- **`use_best_model=True`** — reverts to the best iteration (not the last one).

---
## 12. LightGBM (5-Fold CV)

In [ ]:
oof_lgb = np.full(len(y_all), np.nan)
lgb_fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(splits):
    X_tr_raw = X_all.iloc[train_idx]
    X_va_raw = X_all.iloc[val_idx]
    y_tr = y_all[train_idx]
    y_va = y_all[val_idx]

    # Tree preprocessing (minimal)
    X_tr_lgb, X_va_lgb = prepare_for_trees(X_tr_raw, X_va_raw, final_cat)

    safe_names_lgb = sanitize_names(X_tr_lgb.columns.tolist())
    X_tr_lgb.columns = safe_names_lgb
    X_va_lgb.columns = safe_names_lgb

    # Compute scale_pos_weight for this fold
    pos = float(y_tr.sum())
    neg = float(len(y_tr) - pos)
    spw = neg / pos if pos > 0 else 1.0

    cat_names = [sanitize_names([c])[0] for c in final_cat if c in X_tr_lgb.columns]

    lgbm = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=63,
        objective="binary",
        metric="average_precision",
        scale_pos_weight=spw,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    lgbm.fit(
        X_tr_lgb, y_tr,
        eval_set=[(X_va_lgb, y_va)],
        categorical_feature=cat_names if cat_names else "auto",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )

    val_prob = lgbm.predict_proba(X_va_lgb)[:, 1]
    oof_lgb[val_idx] = val_prob

    pr_auc  = average_precision_score(y_va, val_prob)
    roc_auc = roc_auc_score(y_va, val_prob)
    lgb_fold_metrics.append({"fold": fold_idx, "pr_auc": pr_auc, "roc_auc": roc_auc})
    print(f"  Fold {fold_idx}: PR-AUC={pr_auc:.4f} | ROC-AUC={roc_auc:.4f}")

lgb_metrics_df = pd.DataFrame(lgb_fold_metrics)
print(f"\nLightGBM — mean CV:")
print(f"  PR-AUC:  {lgb_metrics_df['pr_auc'].mean():.4f} ± {lgb_metrics_df['pr_auc'].std():.4f}")
print(f"  ROC-AUC: {lgb_metrics_df['roc_auc'].mean():.4f} ± {lgb_metrics_df['roc_auc'].std():.4f}")

LGB_FEATURE_NAMES = safe_names_lgb

**Why LightGBM:**
- **Fastest boosting algorithm** — histogram-based splitting is very efficient on wide datasets (500+ features).
- **Handles NaN natively** — same advantage as CatBoost.
- **`scale_pos_weight`** — computed per fold as `n_negative / n_positive` (~42×), which tells the model to penalize false negatives much more than false positives.
- **`num_leaves=63`** — allows moderately complex trees; the default (31) might underfit on 500+ features.
- **Different from CatBoost** in how it handles categoricals (LightGBM uses Fisher-split grouping on label-encoded values; CatBoost uses ordered target statistics). Having both gives us a diversity of approaches for comparison.

---
## 13. Model Comparison & Final Plots

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
models = {
    "Logistic Regression": oof_lr,
    "CatBoost":            oof_cb,
    "LightGBM":            oof_lgb,
}

results = []
for name, oof in models.items():
    valid = ~np.isnan(oof)
    y_v = y_all[valid]
    p_v = oof[valid]

    pr_auc  = average_precision_score(y_v, p_v)
    roc_auc = roc_auc_score(y_v, p_v)
    brier   = brier_score_loss(y_v, p_v)
    opt_t, max_f1 = find_optimal_f1(y_v, p_v)

    results.append({
        "Model":              name,
        "PR-AUC":             round(pr_auc, 4),
        "ROC-AUC":            round(roc_auc, 4),
        "Brier Score":        round(brier, 4),
        "Optimal Threshold":  round(opt_t, 3),
        "Max F1":             round(max_f1, 4),
    })

results_df = pd.DataFrame(results).set_index("Model")
print("=" * 70)
print("  FINAL METRICS TABLE (OOF — Out-of-Fold Cross-Validation)")
print("=" * 70)
display(results_df)
print("""
Metric explanations:
  PR-AUC (Average Precision) — PRIMARY metric for imbalanced data.
    Higher = better at finding positives without too many false alarms.
  ROC-AUC — ranking quality (1.0 = perfect, 0.5 = random).
  Brier Score — calibration quality (lower = better).
  Max F1 — best F1-score at the optimal probability threshold.
""")

**Why these metrics:** For imbalanced binary classification (~2% positive rate), accuracy is misleading. PR-AUC is the primary metric because it focuses on how well the model identifies the rare positive class. ROC-AUC measures overall ranking quality. Brier score checks if predicted probabilities are well-calibrated (important for risk scoring). Max F1 shows the best trade-off between precision and recall at the optimal threshold.

In [ ]:
# ── Precision-Recall and ROC Curves ──────────────────────────────────────────
colors = [PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# PR Curves
for (name, oof), color in zip(models.items(), colors):
    valid = ~np.isnan(oof)
    prec, rec, _ = precision_recall_curve(y_all[valid], oof[valid])
    ap = average_precision_score(y_all[valid], oof[valid])
    axes[0].plot(rec, prec, label=f"{name} (AP={ap:.4f})", color=color, lw=2)

baseline = y_all.mean()
axes[0].axhline(baseline, color="gray", ls="--", lw=1, label=f"Baseline ({baseline:.3f})")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curves (OOF)")
axes[0].legend()

# ROC Curves
for (name, oof), color in zip(models.items(), colors):
    valid = ~np.isnan(oof)
    fpr, tpr, _ = roc_curve(y_all[valid], oof[valid])
    auc = roc_auc_score(y_all[valid], oof[valid])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})", color=color, lw=2)

axes[1].plot([0, 1], [0, 1], "--", color="gray", lw=1, label="Baseline")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curves (OOF)")
axes[1].legend()

plt.tight_layout()
plt.show()

**Why both curves:** The PR curve (left) is the most informative for imbalanced data — a good model has a curve that stays high even at high recall. The ROC curve (right) is the standard ranking metric. Comparing both helps us see if a model is good at ranking (ROC) but poor at actually separating positives from negatives at typical operating points (PR).

In [ ]:
# ── Calibration Curves (Reliability Diagrams) ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, oof), color in zip(axes, models.items(), colors):
    valid = ~np.isnan(oof)
    y_v, p_v = y_all[valid], oof[valid]

    frac_pos, mean_pred = calibration_curve(y_v, p_v, n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", color=color, lw=2, label="Model")
    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    ax.set_title(f"{name}\nCalibration Curve")
    ax.set_xlabel("Predicted probability")
    ax.set_ylabel("Actual fraction of positives")
    ax.legend()

plt.suptitle("Reliability Diagrams — how well do predicted probabilities match reality?", y=1.01)
plt.tight_layout()
plt.show()

print("""
Interpretation:
  - If the line is ABOVE the diagonal: model underestimates probabilities.
  - If BELOW: model overestimates.
  - Tree models often rank well but are poorly calibrated
    (isotonic regression can fix this in production).
""")

**Why calibration curves:** In credit risk, we care not just about ranking borrowers but about the actual probability of default. A well-calibrated model with "predicted probability = 5%" should have roughly 5% of those cases actually default. If the model is poorly calibrated, we can apply isotonic regression as a post-processing step.

In [ ]:
# ── F1-Score vs Threshold ────────────────────────────────────────────────────
thresholds = np.linspace(0.001, 0.999, 150)

fig, ax = plt.subplots(figsize=(10, 5))

for (name, oof), color in zip(models.items(), colors):
    valid = ~np.isnan(oof)
    y_v, p_v = y_all[valid], oof[valid]
    f1s = [f1_score(y_v, (p_v >= t).astype(int), zero_division=0) for t in thresholds]
    opt_t, max_f1 = find_optimal_f1(y_v, p_v)

    ax.plot(thresholds, f1s, label=f"{name} (max F1={max_f1:.4f} @ t={opt_t:.3f})",
            color=color, lw=2)
    ax.axvline(opt_t, color=color, ls="--", alpha=0.5)

ax.set_xlabel("Probability Threshold")
ax.set_ylabel("F1-Score")
ax.set_title("F1-Score vs Probability Threshold")
ax.legend()
plt.tight_layout()
plt.show()

print("""
Interpretation:
  - Default sklearn threshold is 0.5, but for imbalanced data the
    optimal threshold is usually much lower.
  - The choice of threshold depends on business logic: whether precision
    or recall is more important.
""")

**Why F1 vs threshold:** The default classification threshold (0.5) is almost never optimal for imbalanced datasets. By sweeping thresholds from 0 to 1, we find where F1-score peaks — this is the best trade-off between precision and recall. In production, the threshold would be chosen based on business costs (cost of missing a default vs. cost of a false alarm).

In [ ]:
# ── Confusion Matrices at Optimal F1 Threshold ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, oof), color in zip(axes, models.items(), colors):
    valid = ~np.isnan(oof)
    y_v, p_v = y_all[valid], oof[valid]
    opt_t, _ = find_optimal_f1(y_v, p_v)
    y_pred = (p_v >= opt_t).astype(int)

    cm = confusion_matrix(y_v, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["gb=0", "gb=1"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{name}\n(threshold={opt_t:.3f})")

plt.suptitle("Confusion Matrices at Optimal F1 Threshold", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Why confusion matrices:** They show the actual counts of True Positives (defaults correctly caught), False Positives (false alarms), False Negatives (missed defaults), and True Negatives. This is the most intuitive way to understand model performance in business terms: how many defaults did we catch, and how many false alarms did we trigger?

In [ ]:
# ── Feature Importance (CatBoost + LightGBM) ────────────────────────────────
# Retrain on first fold for feature importance visualization
train_idx_fi, val_idx_fi = splits[0]
X_tr_raw = X_all.iloc[train_idx_fi]
X_va_raw = X_all.iloc[val_idx_fi]
y_tr_fi  = y_all[train_idx_fi]
y_va_fi  = y_all[val_idx_fi]

# CatBoost importance
X_tr_cb_fi, X_va_cb_fi = prepare_for_trees(X_tr_raw, X_va_raw, final_cat)
X_tr_cb_fi.columns = sanitize_names(X_tr_cb_fi.columns.tolist())
X_va_cb_fi.columns = sanitize_names(X_va_cb_fi.columns.tolist())

cb_fi = CatBoostClassifier(
    iterations=200, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="AUC",
    random_seed=RANDOM_STATE, verbose=0,
    allow_writing_files=False, auto_class_weights="Balanced",
)
train_pool_fi = Pool(X_tr_cb_fi, y_tr_fi, cat_features=cat_feature_indices)
val_pool_fi   = Pool(X_va_cb_fi, y_va_fi, cat_features=cat_feature_indices)
cb_fi.fit(train_pool_fi, eval_set=val_pool_fi, use_best_model=True, early_stopping_rounds=30)

cb_importance = pd.Series(
    cb_fi.get_feature_importance(),
    index=X_tr_cb_fi.columns.tolist()
).sort_values(ascending=True).tail(25)

# LightGBM importance
X_tr_lgb_fi, X_va_lgb_fi = prepare_for_trees(X_tr_raw, X_va_raw, final_cat)
X_tr_lgb_fi.columns = sanitize_names(X_tr_lgb_fi.columns.tolist())
X_va_lgb_fi.columns = sanitize_names(X_va_lgb_fi.columns.tolist())

pos_fi = float(y_tr_fi.sum())
neg_fi = float(len(y_tr_fi) - pos_fi)
spw_fi = neg_fi / pos_fi if pos_fi > 0 else 1.0
cat_names_fi = [sanitize_names([c])[0] for c in final_cat if c in X_tr_lgb_fi.columns]

lgb_fi = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=63,
    objective="binary", scale_pos_weight=spw_fi,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgb_fi.fit(
    X_tr_lgb_fi, y_tr_fi,
    eval_set=[(X_va_lgb_fi, y_va_fi)],
    categorical_feature=cat_names_fi if cat_names_fi else "auto",
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)],
)

lgb_importance = pd.Series(
    lgb_fi.feature_importances_,
    index=X_tr_lgb_fi.columns.tolist()
).sort_values(ascending=True).tail(25)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(16, 9))

cb_importance.plot(kind="barh", ax=axes[0], color=PALETTE["primary"])
axes[0].set_title("CatBoost — Top 25 Feature Importance (Gain)")
axes[0].set_xlabel("Importance")

lgb_importance.plot(kind="barh", ax=axes[1], color=PALETTE["secondary"])
axes[1].set_title("LightGBM — Top 25 Feature Importance (Gain)")
axes[1].set_xlabel("Importance")

plt.tight_layout()
plt.show()

**Why feature importance:** This shows which features the models rely on most. If both CatBoost and LightGBM agree on the top features, those are the most robust predictors. If they disagree, it suggests the features might be interchangeable (correlated), and the models just picked different ones from the same group. This insight guides future feature engineering and domain expert validation.

In [ ]:
# ── Fold-by-Fold Comparison (Bar Charts) ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fold_compare_pr = pd.DataFrame({
    "Logistic Regression": lr_metrics_df["pr_auc"].values,
    "CatBoost":            cb_metrics_df["pr_auc"].values,
    "LightGBM":            lgb_metrics_df["pr_auc"].values,
}, index=[f"Fold {i}" for i in range(N_SPLITS)])

fold_compare_roc = pd.DataFrame({
    "Logistic Regression": lr_metrics_df["roc_auc"].values,
    "CatBoost":            cb_metrics_df["roc_auc"].values,
    "LightGBM":            lgb_metrics_df["roc_auc"].values,
}, index=[f"Fold {i}" for i in range(N_SPLITS)])

fold_compare_pr.plot(kind="bar", ax=axes[0],
                     color=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]],
                     edgecolor="white")
axes[0].set_title("PR-AUC per Fold")
axes[0].set_ylabel("PR-AUC")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(fontsize=9)

fold_compare_roc.plot(kind="bar", ax=axes[1],
                      color=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]],
                      edgecolor="white")
axes[1].set_title("ROC-AUC per Fold")
axes[1].set_ylabel("ROC-AUC")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

**Why fold-by-fold comparison:** Seeing metrics per fold tells us about model stability. If a model has high mean PR-AUC but huge variance across folds, it might be overfitting to specific entity groups. A stable model has consistent performance across all folds. Large fold-to-fold variance could also indicate that some folds have a different data distribution (e.g., certain entity types concentrate in one fold).

In [ ]:
# ── Boxplot: Model Stability Across Folds ────────────────────────────────────
lr_all  = pd.DataFrame({"PR-AUC": lr_metrics_df["pr_auc"],  "ROC-AUC": lr_metrics_df["roc_auc"],  "Model": "LR"})
cb_all  = pd.DataFrame({"PR-AUC": cb_metrics_df["pr_auc"],  "ROC-AUC": cb_metrics_df["roc_auc"],  "Model": "CatBoost"})
lgb_all = pd.DataFrame({"PR-AUC": lgb_metrics_df["pr_auc"], "ROC-AUC": lgb_metrics_df["roc_auc"], "Model": "LightGBM"})
all_metrics = pd.concat([lr_all, cb_all, lgb_all], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=all_metrics, x="Model", y="PR-AUC", ax=axes[0],
            palette=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]])
axes[0].set_title("PR-AUC Across 5 Folds")

sns.boxplot(data=all_metrics, x="Model", y="ROC-AUC", ax=axes[1],
            palette=[PALETTE["accent"], PALETTE["primary"], PALETTE["secondary"]])
axes[1].set_title("ROC-AUC Across 5 Folds")

plt.suptitle("Model Stability — smaller spread = more stable model", y=1.02)
plt.tight_layout()
plt.show()

**Why boxplots:** Boxplots visualize both the central tendency (median) and the spread (IQR) of metrics across folds in a single glance. A tighter box means the model generalizes consistently. A wider box or outlier dots mean the model is sensitive to the particular train/val split, which is a red flag for production deployment.

---
## Conclusions & Recommendations

### Key EDA Findings

| Finding | Modeling Implication |
|---------|---------------------|
| **~2.2% positive rate** | Heavy imbalance → optimize PR-AUC, not accuracy |
| **Panel structure** (~5 rows/ID) | Mandatory StratifiedGroupKFold by ID |
| **MNAR features** | Missing = information → add is_missing indicators for LR |
| **700+ correlated pairs** | Critical for LR (multicollinearity) → remove |r|>0.95 pairs |
| **High skewness** (median ~9) | LR needs RobustScaler; trees are unaffected |

### Why Different Preprocessing for LR vs Trees

| Step | Logistic Regression | CatBoost / LightGBM |
|------|--------------------|-----------------------|
| MNAR indicators | ✅ Added (can't learn from NaN) | ❌ Not needed (handles NaN natively) |
| Imputation | ✅ Median (required — LR crashes on NaN) | ❌ Not needed |
| Scaling | ✅ RobustScaler (L2 penalty needs it) | ❌ Not needed (scale-invariant) |
| Label encoding | ✅ Yes | ✅ Yes |

### Model Results (OOF CV)

| Metric | Logistic Regression | CatBoost | LightGBM |
|--------|--------------------|-----------|-----------|
| PR-AUC | ≈ lower | ≈ higher | ≈ higher |
| ROC-AUC | ≈ good | ≈ better | ≈ better |
| Speed | ⚡ Very fast | 🐢 Slower | ⚡ Fast |
| Interpretability | ✅ High | ⚠️ Medium | ⚠️ Medium |

### Recommended Production Model: **CatBoost or LightGBM**

### Next Steps

1. Hyperparameter tuning via `Optuna` or `GridSearchCV`
2. Ensemble (average CatBoost + LightGBM probabilities)
3. SHAP analysis for model explainability
4. Isotonic calibration of probabilities
5. Choose operational threshold based on cost matrix (cost of missing a default vs. false alarm)